<a href="https://colab.research.google.com/github/CS-Edwards/qol_exp_demo/blob/main/QLE_Environmental_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Quality of Life Explorer: Exploring Tree Canopy and Electricity Data Exercise

In this demo, we explore the relationship between **Tree Canopy** and **Electricity Consumption** across neighborhoods using real-world data from the Quality of Life Data Explorer. We combine **tabular attribute data** with **spatial data** to create maps, scatter plots, and bivariate visualizations.

We use the following libraries:

* **`pandas` (`pd`)** – for loading, cleaning, and subsetting tabular data.  
* **`geopandas` (`gpd`)** – for working with spatial (GeoJSON) data and creating choropleth maps.  
* **`matplotlib.pyplot` (`plt`)** – for plotting figures, arranging subplots, and adding titles, labels, and legends.  
* **`matplotlib.colors` (`mcolors`)** – for customizing color schemes in visualizations.  
* **`numpy` (`np`)** – for numerical computations, including trend lines and regression fits.

This setup allows us to integrate spatial and attribute data, explore patterns, and visualize relationships for further analysis.

## RUN ALL CELLS

▶️ **Tip:** To run all cells at once, go to **Runtime → Run all**, or press **Ctrl + F9 (Cmd + F9 on Mac)**.

In [ ]:
!pip install geopandas folium matplotlib --quiet

In [ ]:
# Import All Libraries


import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np


## Subsetting Downloaded Quality of Life Explorer Data

For this demo, we use a subset of the Quality of Life Explorer data download. The dataset has been filtered to include only **Tree Canopy** and **Electricity Consumption** variables, and the resulting subset is saved as a .csv file.
Note: This code is provided for transparency and documentation purposes only and should not be executed.


```
# Load Data

# Load .csv DataDownload| Quality of Life Explorere
# df = pd.read_csv("//content//Data_Download_20240305.csv")

#Explore Data

# print(df['data_year'].unique())
# print(df[df['Normalized_Data_Name'] == 'Tree_Canopy']['data_year'].unique())

# Explore datafame

# df.head()
# df.info()


# Create subset of containing only 'Treec Canopy' and 'Electricity_Consumption' data

# data_name_1 = "Tree_Canopy"
# data_name_2 = "Electricity_Consumption"

# filtered = df[df['Normalized_Data_Name'].isin([
#     data_name_1,
#     data_name_2
#     ])]


# filtered.shape
# filtered.head()

# Write filtered data to .csv
#filtered.to_csv("canopy_energy_sample_data.csv", index=False)
```



## Tree Canopy Summary Statistics

In this section, we explore the Tree Canopy attribute data from the Quality of Life `data_download.csv`. We begin by examining the dataset and computing summary statistics to identify NPAs with the highest and lowest tree canopy coverage, followed by visualizing these metrics.



In [ ]:
#Import Tree Canopy and Energy Consumption - Electric CSV Data

url = "https://raw.githubusercontent.com/CS-Edwards/qol_exp_demo/refs/heads/main/sample_data/canopy_energy_sample_data.csv"
data_df = pd.read_csv(url)

In [ ]:
data_df.info()

In [ ]:
data_df.head()

In [ ]:
#Summary Statistics for Tree Canopy

canopy_df = data_df[data_df["Normalized_Data_Name"] == "Tree_Canopy"]

avg_canopy = canopy_df['normalized'].mean()
median_canopy = canopy_df['normalized'].median()
skew_canopy = canopy_df['normalized'].skew()
std_canopy = canopy_df['normalized'].std()
q25 = canopy_df['normalized'].quantile(0.25)
q75 = canopy_df['normalized'].quantile(0.75)
iqr = q75 - q25
worst_npa = canopy_df.loc[canopy_df['normalized'].idxmin(), 'NPA']
worst_val = canopy_df['normalized'].min()
best_npa = canopy_df.loc[canopy_df['normalized'].idxmax(), 'NPA']
best_val = canopy_df['normalized'].max()

skew_direction = "right (more low-canopy NPAs)" if skew_canopy > 0 else "left (more high-canopy NPAs)"

print(f"""
=== Tree Canopy Coverage Summary ===
Central Tendency:
  Mean:   {avg_canopy:.2f}%
  Median: {median_canopy:.2f}%

Spread:
  Std Dev: {std_canopy:.2f}%
  IQR:     {iqr:.2f}% (25th: {q25:.2f}%, 75th: {q75:.2f}%)
  Skew:    {skew_canopy:.2f} — skewed {skew_direction}

Extremes:
  Lowest coverage:  NPA {worst_npa} at {worst_val:.2f}%
  Highest coverage: NPA {best_npa} at {best_val:.2f}%
""")

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Tree Canopy Coverage Summary', fontsize=16, fontweight='bold')

# 1. Histogram — distribution shape
axes[0,0].hist(canopy_df['normalized'], bins=20, edgecolor='black', color='forestgreen', alpha=0.7)
axes[0,0].axvline(avg_canopy, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_canopy:.2f}%')
axes[0,0].axvline(median_canopy, color='blue', linestyle='--', linewidth=2, label=f'Median: {median_canopy:.2f}%')
axes[0,0].set_xlabel('% Land Area Covered by Tree Canopy')
axes[0,0].set_ylabel('Number of NPAs')
axes[0,0].set_title('Distribution of Tree Canopy Coverage')
axes[0,0].legend()

# 2. Boxplot — spread and outliers
axes[0,1].boxplot(canopy_df['normalized'], vert=False, patch_artist=True,
                  boxprops=dict(facecolor='forestgreen', alpha=0.7))
axes[0,1].axvline(avg_canopy, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_canopy:.2f}%')
axes[0,1].set_xlabel('% Land Area Covered by Tree Canopy')
axes[0,1].set_title('Spread and Outliers')
axes[0,1].legend()

#3. Top 5 and Bottom 5 NPAs — extremes
top5 = canopy_df.nlargest(5, 'normalized')
bottom5 = canopy_df.nsmallest(5, 'normalized')
extremes = pd.concat([bottom5, top5])
colors = ['tomato'] * 5 + ['forestgreen'] * 5

axes[1,0].barh(extremes['NPA'].astype(str), extremes['normalized'],
               color=colors, edgecolor='black')
axes[1,0].axvline(avg_canopy, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_canopy:.2f}%')
axes[1,0].set_xlabel('% Land Area Covered by Tree Canopy')
axes[1,0].set_title('Top 5 and Bottom 5 NPAs')
axes[1,0].legend()

# 4. Equity categories
bins = [0, 20, 40, 60, 100]
labels = ['Low\n(<20%)', 'Moderate\n(20-40%)', 'Good\n(40-60%)', 'High\n(>60%)']
canopy_df['category'] = pd.cut(canopy_df['normalized'], bins=bins, labels=labels)
counts = canopy_df['category'].value_counts().reindex(labels)

axes[1,1].bar(labels, counts, color=['tomato', 'orange', 'yellowgreen', 'forestgreen'],
              edgecolor='black')
axes[1,1].set_xlabel('Coverage Category')
axes[1,1].set_ylabel('Number of NPAs')
axes[1,1].set_title('Tree Canopy Equity Categories')

plt.tight_layout()
plt.show()

#Spatial Data GeoJSON

In this section we import and explore our spatial data sourced from the Quality of Life Explorer file zip.


In [ ]:
# Import GeoJSON Data
url = "https://raw.githubusercontent.com/CS-Edwards/qol_exp_demo/refs/heads/main/sample_data/tree_canopy.geojson"
gdf = gpd.read_file(url)


In [ ]:
gdf.head()

In [ ]:
gdf.plot()

## Merge and Interdomain Visualization:

This section merges spatial (GeoJSON) data with tabular attribute data to enable mapping. It produces side-by-side maps of tree canopy and electricity consumption, as well as a bivariate analysis map showing a possible relationship between the two variables.

In [ ]:
# Inspect the GeoDataFrame, notice the Dtype (data type) of the 'id' is a string/object
gdf.info()

In [ ]:
# Change the data type to int (integer)
gdf["id"] = gdf["id"].astype(int)
gdf.info()

In [ ]:
print(data_df['data_year'].unique())

In [ ]:
# Dropping 2013 Electric data since we only have tree canopy data for 2012
data_2012 = data_df[data_df['data_year']==2012]

In [ ]:
# Debug: check how many NPA IDs exist in BOTH the GeoJSON and the data
# This helps confirm the merge will work and shows if there are mismatches
common = set(gdf["id"]) & set(data_2012["NPA"])
print(len(common))

In [ ]:
# Merge attribute data (csv/dataframe) with geospatial data, on the "id" and "NPA" columns
merged = gdf.merge(data_2012, left_on = "id", right_on="NPA", how = "left")
merged.head()


In [ ]:
merged.info()

In [ ]:
# Pivot to wide format
merged_wide = merged.pivot_table(
    index=['NPA', 'geometry'],
    columns='Normalized_Data_Name',
    values='normalized'
).reset_index()

# Clean up column names
merged_wide.columns.name = None

# Convert back to GeoDataFrame
merged_wide = gpd.GeoDataFrame(merged_wide, geometry='geometry')

print(merged_wide.columns.tolist())
print(f"Rows: {len(merged_wide)}")
print(merged_wide[['NPA', 'Tree_Canopy', 'Electricity_Consumption']].head())

In [ ]:
merged_wide.head()

In [ ]:

fig = plt.figure(figsize=(18, 16))


# 1. Side by side choropleth maps
ax1 = fig.add_subplot(2, 2, 1)
ax2 = fig.add_subplot(2, 2, 2)

merged_wide.plot(column='Tree_Canopy', ax=ax1, cmap='Greens',
                 legend=True, edgecolor='black', linewidth=0.5)
ax1.set_title('Tree Canopy Coverage\n(% of land area)', fontsize=12, fontweight='bold')
ax1.axis('off')


# merged_wide.plot(column='Electricity_Consumption', ax=ax2, cmap='RdYlGn_r',   #reverse cmap color scale
#                  legend=True, edgecolor='black', linewidth=0.5)
merged_wide.plot(column='Electricity_Consumption', ax=ax2, cmap='RdYlGn',
                 legend=True, edgecolor='black', linewidth=0.5)
ax2.set_title('Electricity Consumption\n(avg kWh/month per household)', fontsize=12, fontweight='bold')
ax2.axis('off')


# 2. Scatter plot — canopy vs electricity
ax3 = fig.add_subplot(2, 2, 3)

ax3.scatter(merged_wide['Tree_Canopy'], merged_wide['Electricity_Consumption'],
            alpha=0.6, edgecolor='black', color='forestgreen')


# Drop NaNs from both columns together
clean = merged_wide[['Tree_Canopy', 'Electricity_Consumption']].dropna()

z = np.polyfit(clean['Tree_Canopy'], clean['Electricity_Consumption'], 1)
p = np.poly1d(z)
x_line = np.linspace(clean['Tree_Canopy'].min(), clean['Tree_Canopy'].max(), 100)
ax3.plot(x_line, p(x_line), color='red', linestyle='--', linewidth=2, label='Trend')

# Add a trend line
# z = np.polyfit(merged_wide['Tree_Canopy'].dropna(),
#                merged_wide['Electricity_Consumption'].dropna(), 1)
# p = np.poly1d(z)
# x_line = np.linspace(merged_wide['Tree_Canopy'].min(), merged_wide['Tree_Canopy'].max(), 100)
# ax3.plot(x_line, p(x_line), color='red', linestyle='--', linewidth=2, label='Trend')

ax3.set_xlabel('Tree Canopy Coverage (%)')
ax3.set_ylabel('Electricity Consumption (kWh/month)')
ax3.set_title('Tree Canopy vs Electricity Consumption\nby NPA', fontsize=12, fontweight='bold')
ax3.legend()


# 3. Bivariate map — both variables in one map
ax4 = fig.add_subplot(2, 2, 4)

# Classify each variable into 3 quantile buckets
merged_wide['canopy_q'] = pd.qcut(merged_wide['Tree_Canopy'], q=3, labels=[1, 2, 3])
merged_wide['elec_q'] = pd.qcut(merged_wide['Electricity_Consumption'], q=3, labels=[1, 2, 3])

# Create bivariate class (9 combinations)
merged_wide['bivariate'] = merged_wide['canopy_q'].astype(str) + merged_wide['elec_q'].astype(str)

# Color scheme — rows=canopy (low to high), cols=electricity (low to high)
bivariate_colors = {
    '11': '#e8f4d4',  # low canopy, low electricity
    '12': '#b8d9a0',
    '13': '#4a9e5c',  # low canopy, high electricity
    '21': '#f3d79a',
    '22': '#c9a86c',
    '23': '#8c6c3e',
    '31': '#f7a58a',  # high canopy, low electricity
    '32': '#d4716b',
    '33': '#b02020',  # high canopy, high electricity
}

#merged_wide['color'] = merged_wide['bivariate'].map(bivariate_colors)
merged_wide['color'] = merged_wide['bivariate'].map(bivariate_colors).fillna('#d3d3d3')
merged_wide.plot(ax=ax4, color=merged_wide['color'], edgecolor='black', linewidth=0.5)
ax4.set_title('Bivariate Map\nTree Canopy vs Electricity Consumption', fontsize=12, fontweight='bold')
ax4.axis('off')

# Simple legend for bivariate map
legend_labels = {
    '#e8f4d4': 'Low Canopy, Low Electricity',
    '#4a9e5c': 'Low Canopy, High Electricity',
    '#f7a58a': 'High Canopy, Low Electricity',
    '#b02020': 'High Canopy, High Electricity',
}

# Documentation: https://matplotlib.org/stable/api/_as_gen/matplotlib.patches.Patch.html
# Custom legend entries (colored boxes) for the bivariate map
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, edgecolor='black', label=label)
                   for color, label in legend_labels.items()]
ax4.legend(handles=legend_elements, loc='lower left', fontsize=7)

plt.suptitle('Charlotte NPA — Tree Canopy & Electricity Consumption (2012)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Next Steps/ Exercises:

*   Continue to explore the relationship between Tree Canopy and Electricity using data comparison visualisations...
*   Subset `data_df` to create a dedicated dataframe for Electricity Consumption and compute summary statistics...
* Revisit the [Quality of Life Data Explorer](https://ui.charlotte.edu/our-work/quality-life-explorer) to investigate additional intersections between variables of interest...

